In [ ]:
import os
from io import BytesIO
from dotenv import load_dotenv
import requests

import pandas as pd

In [ ]:
# Load environment parameters, .env file should be in them same directory level with this notebook fiel as per the README.md
load_dotenv()

SITE = os.environ["SITE"]
DOMAIN = os.environ["DOMAIN"]

FEDAUTH = os.environ["FEDAUTH"]
RTFA = os.environ["RTFA"]

SITE_PATH = os.environ["SITE_PATH"]
FILE_PATH = os.environ["FILE_PATH"]

In [ ]:
def download_sharepoint_file(
    *,
    site: str,
    domain: str,
    fedauth: str,
    rtfa: str,
    site_path: str,
    file_path: str,
    timeout: int = 30,
) -> bytes:
    """
    Download a file from a SharePoint personal site using an existing
    authenticated SharePoint browser session.

    Authentication
    --------------
    This function uses the SharePoint authentication cookies `FedAuth`
    and `rtFa`. These values must be supplied explicitly by the caller.

    Parameters
    ----------
    site : str
        SharePoint host, including scheme.

        Example:
            "https://connectqutedu-my.sharepoint.com"

    domain : str
        Cookie domain for the SharePoint host.

        Example:
            "connectqutedu-my.sharepoint.com"

    fedauth : str
        Value of the `FedAuth` authentication cookie.

        IMPORTANT: Treat this as a credential. Do not commit it to
        source code or print it.

    rtfa : str
        Value of the `rtFa` authentication cookie.

        IMPORTANT: Treat this as a credential. Do not commit it to
        source code or print it.

    site_path : str
        Server-relative path of the SharePoint site.

        Example:
            "/personal/n12554561_qut_edu_au"

    file_path : str
        Server-relative path of the file.

        Example:
            "/personal/n12554561_qut_edu_au/"
            "Documents/Group25 Overpriced Rooms, Underslept Minds.xlsx"

    timeout : int, default=30
        HTTP request timeout in seconds.

    Returns
    -------
    bytes
        Raw contents of the requested file.

    Raises
    ------
    requests.HTTPError
        If SharePoint returns an unsuccessful HTTP status.

    Notes
    -----
    The SharePoint REST endpoint used is:

        /_api/Web/GetFileByServerRelativePath(
            decodedurl='<file_path>'
        )/$value

    The returned bytes are not interpreted by this function. For example,
    an XLSX workbook can subsequently be passed to pandas/openpyxl.

    The `FedAuth` and `rtFa` cookies are installed only in this local
    requests.Session() and are not persisted by the function.
    """

    # Normalize inputs
    site = site.rstrip("/")
    site_path = "/" + site_path.strip("/")

    if not file_path.startswith("/"):
        file_path = "/" + file_path

    # Construct the SharePoint REST endpoint.
    rest_url = (
        f"{site}{site_path}"
        "/_api/Web/GetFileByServerRelativePath("
        f"decodedurl='{file_path}'"
        ")/$value"
    )

    # Create an isolated HTTP session.
    session = requests.Session()

    # SharePoint authentication cookies.
    session.cookies.set(
        "FedAuth",
        fedauth,
        domain=domain,
        path="/",
    )

    session.cookies.set(
        "rtFa",
        rtfa,
        domain=domain,
        path="/",
    )

    # Download the actual file contents.
    response = session.get(
        rest_url,
        timeout=timeout,
        allow_redirects=False,
    )

    response.raise_for_status()

    return response.content

In [ ]:
def load_excel_sheet(file_bytes: bytes, sheet_name: str = "Sheet1"):
    """
    Load one worksheet from an XLSX byte stream into a pandas DataFrame.
    """
    return pd.read_excel(
        BytesIO(file_bytes),
        sheet_name=sheet_name,
    )

In [ ]:
file_bytes = download_sharepoint_file(
    site=SITE,
    domain=DOMAIN,
    fedauth=FEDAUTH,
    rtfa=RTFA,
    site_path=SITE_PATH,
    file_path=FILE_PATH,
)

df = load_excel_sheet(file_bytes, sheet_name="Sheet1")

In [ ]:
df.columns